# 74. The Low-Energy Neutral Atom Imager for IMAGE — Implementation / IMAGE LENA 구현

**Paper**: Moore, T. E., et al., "The Low-Energy Neutral Atom Imager for IMAGE", *Space Sci. Rev.* **91**, 155-195, 2000. DOI: 10.1023/A:1005211509003

This notebook reproduces the key algorithms and operating principles of the LENA imager: (1) atom-to-negative-ion **surface conversion efficiency** for 10–750 eV neutrals, (2) the **polar-wind / cleft outflow imaging geometry**, (3) **H/O composition mapping** during a storm, and (4) the **low-energy noise model** including geocoronal Lyman-α dayglow rejection.

이 노트북은 LENA 이미저의 핵심 알고리즘을 재현한다: (1) 10–750 eV 중성원자에 대한 **표면 변환 효율(atom-to-negative-ion)**, (2) **polar wind / cleft outflow 영상화 기하**, (3) 폭풍 동안의 **H/O 조성 매핑**, (4) **저에너지 잡음 모델** 및 dayglow 차단.

In [ ]:
"""Imports and global plotting setup.

Uses the study-with-ai conda environment (NumPy, SciPy, Matplotlib).
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy import ndimage
from scipy.special import erf

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

RNG = np.random.default_rng(seed=20000401)

# Physical constants in SI / cgs as needed.
EV_TO_J = 1.602176634e-19  # 1 eV in joules
AMU_TO_KG = 1.66053907e-27  # 1 amu in kilograms
R_E_KM = 6371.2  # Earth radius (km)

## Part 1: Atom-to-Negative-Ion Surface Conversion Efficiency / 표면 변환 효율

LENA replaces classical thin-foil ENA detection with **surface conversion** on a polished, –20 kV biased polycrystalline tungsten surface naturally coated with a monolayer of adsorbed water. Incoming neutrals strike at 75° from the surface normal and are near-specularly reflected; a fraction `eta(E)` emerges as a negative ion with retained energy `<E_t> ~ alpha * E_inc` (alpha_H ~ 0.80, alpha_O ~ 0.60).

We model `eta(E)` as a power-law (consistent with Fig. 12 of the paper) and energy retention as a Gaussian centered at `alpha * E_inc` with FWHM equal to the mean (broad inelastic distribution).

LENA는 폴드 W 표면에서 입사 중성원자를 **음이온으로 변환**하여 검출한다. 변환 효율 η(E)는 입사 에너지에 따라 power-law로 증가하며, H는 입사 에너지의 ~80%, O는 ~60%를 보존한다.

In [ ]:
def conversion_efficiency(energy_ev: np.ndarray, species: str = "H") -> np.ndarray:
    """Model surface conversion efficiency vs. incident energy.

    Power-law fit consistent with Moore et al. (2000) Figure 12: at 1 keV,
    eta_O ~ a few x 10^-3 and eta_H ~ 10^-3, falling roughly linearly toward
    10^-5 at 30 eV.

    Args:
        energy_ev: Incident neutral kinetic energy in eV.
        species: Either "H" or "O".

    Returns:
        Conversion efficiency eta = (negative ions out) / (neutrals in).
    """
    e = np.asarray(energy_ev, dtype=float)
    if species == "H":
        eta_1kev, beta = 1.0e-4, 1.5
    elif species == "O":
        eta_1kev, beta = 3.0e-3, 1.5
    else:
        raise ValueError(f"Unknown species: {species}")
    return eta_1kev * (e / 1000.0) ** beta


def energy_retention_pdf(
    e_out_ev: np.ndarray, e_in_ev: float, species: str = "H"
) -> np.ndarray:
    """Distribution of reflected negative-ion energy for fixed incident energy.

    Modeled as a Gaussian centered at alpha * E_in with FWHM ~ alpha * E_in,
    truncated at zero (broad inelastic peak; Moore 2000 §4.1.2).
    """
    alpha = {"H": 0.80, "O": 0.60}[species]
    mu = alpha * e_in_ev
    sigma = mu / 2.355  # FWHM ~ mean
    pdf = np.exp(-0.5 * ((e_out_ev - mu) / sigma) ** 2)
    pdf = np.where(e_out_ev > 0, pdf, 0.0)
    norm = np.trapz(pdf, e_out_ev)
    return pdf / norm if norm > 0 else pdf


energy_axis = np.logspace(np.log10(10), np.log10(750), 200)
eta_H = conversion_efficiency(energy_axis, "H")
eta_O = conversion_efficiency(energy_axis, "O")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].loglog(energy_axis, eta_H, label="H atoms / 수소", lw=2)
axes[0].loglog(energy_axis, eta_O, label="O atoms / 산소", lw=2)
axes[0].set_xlabel("Incident energy [eV] / 입사 에너지")
axes[0].set_ylabel(r"Conversion efficiency $\eta$ / 변환 효율")
axes[0].set_title("LENA conversion surface response (Fig. 12 model) / LENA 변환 표면 응답")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()

e_out_axis = np.linspace(0, 800, 400)
for e_in, color in [(100, "C0"), (300, "C1"), (600, "C2")]:
    axes[1].plot(
        e_out_axis, energy_retention_pdf(e_out_axis, e_in, "H"),
        ls="-", color=color, label=f"H, E_in = {e_in} eV"
    )
    axes[1].plot(
        e_out_axis, energy_retention_pdf(e_out_axis, e_in, "O"),
        ls="--", color=color, label=f"O, E_in = {e_in} eV"
    )
axes[1].set_xlabel("Reflected energy [eV] / 반사 에너지")
axes[1].set_ylabel("PDF")
axes[1].set_title("Energy retention upon reflection / 반사 에너지 보존 분포")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"At 100 eV: eta_H = {conversion_efficiency(100, 'H'):.2e}, "
      f"eta_O = {conversion_efficiency(100, 'O'):.2e}")
print(f"At 750 eV: eta_H = {conversion_efficiency(750, 'H'):.2e}, "
      f"eta_O = {conversion_efficiency(750, 'O'):.2e}")

## Part 1b: Effective Area From Conversion Efficiency / 변환 효율로부터 유효 면적

LENA's effective area is the geometric aperture multiplied by all serial transmission and detection probabilities:

$$A_{\rm eff}(E) = A_{\rm geo}\cdot \eta(E) \cdot T_{\rm IXL}\cdot T_{\rm ESA}\cdot \epsilon_{\rm foil}\cdot \epsilon_{\rm MCP}$$

We adopt nominal sub-stage transmissions and reproduce the order-of-magnitude curve from Fig. 12: 10⁻⁵ cm² at 30 eV up to ~10⁻³ cm² at 1 keV for O.

유효 면적은 기하 aperture × η × IXL/ESA 투과율 × foil/MCP 검출 효율의 곱이다.

In [ ]:
def effective_area(energy_ev: np.ndarray, species: str = "H") -> np.ndarray:
    """Effective area of LENA per pixel.

    Args:
        energy_ev: Incident neutral energy in eV.
        species: "H" or "O".

    Returns:
        Effective area in cm^2 per pixel.
    """
    a_geo_cm2 = 1.0  # 1 cm^2 per pixel physical aperture (Table I)
    t_ixl = 0.5  # IXL transmission
    t_esa = 0.4  # ESA truncated hemisphere transmission
    eps_foil = 0.5  # 2 ug/cm^2 foil + secondary-electron yield (energy-dep, simplified)
    eps_mcp = 0.6  # MCP coincidence detection efficiency
    return (
        a_geo_cm2
        * conversion_efficiency(energy_ev, species)
        * t_ixl * t_esa * eps_foil * eps_mcp
    )


fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(energy_axis, effective_area(energy_axis, "H"), label="H", lw=2)
ax.loglog(energy_axis, effective_area(energy_axis, "O"), label="O", lw=2)
ax.set_xlabel("Incident energy [eV]")
ax.set_ylabel(r"$A_{\rm eff}$ [cm$^2$]")
ax.set_title("LENA effective area (Fig. 12 reproduction) / LENA 유효 면적")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Part 2: Polar-Wind / Cleft Outflow Imaging Geometry / Polar Wind 및 Cleft Outflow 영상화 기하

From IMAGE apogee (8 R_E) LENA looks back at the polar ionosphere. We construct a synthetic ionospheric outflow source map (auroral oval + cleft + polar cap) and project it onto the LENA 12 (polar) × 45 (azimuth) pixel grid.

We model the source as a sum of three regions:
* **Auroral oval**: ring at magnetic latitude 65°–75° with 30 eV characteristic energy.
* **Cleft**: localized noon-sector outflow at MLAT 75°–80°, MLT 10–14, with 50 eV.
* **Polar cap**: weak diffuse outflow above 80° MLAT at 20 eV.

IMAGE 위성 원지점(8 R_E)에서 LENA가 극지 전리권을 후방 영상화한다. Auroral oval, cleft, polar cap 세 영역의 합으로 source를 모델링한다.

In [ ]:
def synthetic_outflow_map(
    n_mlat: int = 90, n_mlt: int = 180, kp: float = 4.0
) -> dict:
    """Build a synthetic ionospheric ion outflow source map.

    Args:
        n_mlat: Number of magnetic-latitude bins (50 deg .. 90 deg, sampled every 0.44 deg).
        n_mlt: Number of magnetic-local-time bins (0..24 h).
        kp: Geomagnetic activity index used to scale the auroral component.

    Returns:
        Dictionary with mlat (deg), mlt (hr), and per-region flux maps in cm^-2 s^-1.
    """
    mlat = np.linspace(50.0, 90.0, n_mlat)
    mlt = np.linspace(0.0, 24.0, n_mlt, endpoint=False)
    mlat_g, mlt_g = np.meshgrid(mlat, mlt, indexing="ij")

    # Auroral oval: Gaussian ring centered at 70 deg MLAT, broadened at midnight.
    midnight_offset = 5.0 * np.cos((mlt_g - 24.0) * np.pi / 12.0)
    oval_lat0 = 70.0 - 0.05 * midnight_offset  # poleward shift on dayside
    sigma_lat = 3.0 + 1.5 * np.cos(mlt_g * np.pi / 12.0)  # narrower on dayside
    auroral = np.exp(-0.5 * ((mlat_g - oval_lat0) / sigma_lat) ** 2)
    auroral *= 1e8 * (1 + 0.5 * (kp - 3))  # Yau et al. (1988) scaling

    # Cleft: localized near noon, 75-80 deg MLAT.
    cleft = np.exp(-0.5 * ((mlat_g - 78.0) / 1.5) ** 2) * np.exp(
        -0.5 * (((mlt_g - 12.0)) / 1.0) ** 2
    ) * 5e8

    # Polar cap: diffuse, above 80 deg MLAT.
    polar_cap = np.where(mlat_g > 80.0, 1e7, 0.0)

    return {
        "mlat": mlat,
        "mlt": mlt,
        "auroral": auroral,
        "cleft": cleft,
        "polar_cap": polar_cap,
        "total": auroral + cleft + polar_cap,
    }


outflow = synthetic_outflow_map(kp=6.0)

fig, ax = plt.subplots(figsize=(9, 5), subplot_kw={"projection": "polar"})
theta = outflow["mlt"] * np.pi / 12.0
r = 90.0 - outflow["mlat"]
th_g, r_g = np.meshgrid(theta, r, indexing="xy")
im = ax.pcolormesh(
    th_g, r_g, outflow["total"].T, norm=LogNorm(vmin=1e6, vmax=1e9), cmap="viridis"
)
ax.set_theta_zero_location("S")
ax.set_theta_direction(-1)
ax.set_rmax(40)
ax.set_rticks([10, 20, 30, 40])
ax.set_rlabel_position(135)
ax.set_xticks(np.linspace(0, 2 * np.pi, 8, endpoint=False))
ax.set_xticklabels(["00", "03", "06", "09", "12", "15", "18", "21"])
ax.set_title("Synthetic ionospheric outflow source (Kp=6) / 합성 전리권 outflow source")
fig.colorbar(im, ax=ax, label=r"Flux [cm$^{-2}$ s$^{-1}$]")
plt.tight_layout()
plt.show()

### Project source onto LENA pixel grid / Source를 LENA 픽셀 grid에 투영

From the IMAGE apogee, the polar ionosphere subtends roughly the central 90° polar × 360° azimuth section of LENA's spin-built image. We bin the source map onto LENA's 12 polar × 45 azimuth pixels and add the convolution by the polar PSF (Gaussian, FWHM ≈ 2.5 bins).

12 × 45 픽셀로 binning 후 polar PSF (FWHM ≈ 2.5 bins) 콘볼루션.

In [ ]:
def project_to_lena_pixels(
    outflow_map: dict, n_polar: int = 12, n_az: int = 45
) -> np.ndarray:
    """Project ionospheric source onto the LENA 12x45 pixel grid.

    Args:
        outflow_map: Output from synthetic_outflow_map.
        n_polar: Number of polar pixels (default 12, matches Table I).
        n_az: Number of azimuth pixels per spin (default 45 = 360 deg / 8 deg).

    Returns:
        2D array of shape (n_polar, n_az) with linear flux units.
    """
    src = outflow_map["total"]
    mlat = outflow_map["mlat"]
    mlt = outflow_map["mlt"]

    # Polar pixel: maps 50-90 deg MLAT into 12 bins
    pol_bins = np.linspace(mlat.min(), mlat.max(), n_polar + 1)
    az_bins = np.linspace(0.0, 24.0, n_az + 1)

    img = np.zeros((n_polar, n_az))
    for i in range(n_polar):
        i_lat = (mlat >= pol_bins[i]) & (mlat < pol_bins[i + 1])
        for j in range(n_az):
            j_mlt = (mlt >= az_bins[j]) & (mlt < az_bins[j + 1])
            mask = np.outer(i_lat, j_mlt)
            if mask.any():
                img[i, j] = src[mask].mean()
    # Polar PSF smearing (FWHM ~ 2.5 bins)
    img_blur = ndimage.gaussian_filter(img, sigma=(2.5 / 2.355, 0.5))
    return img_blur


lena_image = project_to_lena_pixels(outflow)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(
    lena_image, aspect="auto", origin="lower",
    norm=LogNorm(vmin=1e6, vmax=lena_image.max()), cmap="viridis"
)
ax.set_xlabel("Azimuth pixel (spin phase) / 방위각 픽셀")
ax.set_ylabel("Polar pixel / 극각 픽셀")
ax.set_title("LENA pixelized image of polar outflow / LENA 픽셀 영상")
fig.colorbar(im, ax=ax, label="Source flux")
plt.tight_layout()
plt.show()

## Part 3: H/O Composition Mapping During a Storm / 폭풍 중 H/O 조성 매핑

The ITOF (imaging time-of-flight) module separates H from O via the relation

$$t_{\rm TOF} = L\sqrt{\frac{m}{2qV_{\rm post}}}$$

with `V_post = 20 kV` and a flight path `L ≈ 6 cm`. We simulate the TOF spectrum, apply the species-mapping algorithm, and produce H and O image cubes for the Kp=6 storm condition.

ITOF는 비행 시간으로 m/q를 결정한다. 20 kV post-acceleration에서 H/O 비행 시간 비율은 4.

In [ ]:
def tof_ns(mass_amu: float, energy_ev: float, l_cm: float = 6.0,
           v_post_kv: float = 20.0) -> float:
    """Time of flight for ions through the ITOF stage.

    Includes the post-acceleration potential V_post (negative ion gains qV_post on
    top of its retained kinetic energy).

    Args:
        mass_amu: Particle mass in atomic mass units.
        energy_ev: Kinetic energy at foil entrance in eV.
        l_cm: Flight path in cm.
        v_post_kv: Post-acceleration potential in kV.

    Returns:
        Time of flight in nanoseconds.
    """
    m_kg = mass_amu * AMU_TO_KG
    e_total_j = (energy_ev + 1000.0 * v_post_kv) * EV_TO_J
    v = np.sqrt(2.0 * e_total_j / m_kg)
    return 1e9 * (l_cm * 1e-2) / v


print(f"H+ at 100 eV (post-accel 20 kV): t = {tof_ns(1, 100):.2f} ns")
print(f"O+ at 100 eV (post-accel 20 kV): t = {tof_ns(16, 100):.2f} ns")
print(f"Ratio t_O / t_H = {tof_ns(16, 100) / tof_ns(1, 100):.2f}")

In [ ]:
def simulate_tof_spectrum(
    n_h_events: int = 5000, n_o_events: int = 8000,
    energy_ev: float = 100.0, foil_jitter_ns: float = 0.6,
) -> np.ndarray:
    """Simulate an ITOF spectrum given a known species mix.

    Foil energy/angle straggling is approximated by Gaussian jitter on TOF.

    Args:
        n_h_events: Number of H events.
        n_o_events: Number of O events.
        energy_ev: Pre-foil kinetic energy in eV.
        foil_jitter_ns: 1-sigma TOF spread from foil straggling.

    Returns:
        Array of TOF values in ns for all events.
    """
    t_h = tof_ns(1, energy_ev)
    t_o = tof_ns(16, energy_ev)
    h = RNG.normal(t_h, foil_jitter_ns, n_h_events)
    o = RNG.normal(t_o, 1.5 * foil_jitter_ns, n_o_events)
    return np.concatenate([h, o])


tof_events = simulate_tof_spectrum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(tof_events, bins=200, color="steelblue", edgecolor="black", alpha=0.7)
ax.axvline(tof_ns(1, 100), color="crimson", ls="--", label="H+ expected / 예상")
ax.axvline(tof_ns(16, 100), color="darkorange", ls="--", label="O+ expected / 예상")
ax.set_xlabel("Time of flight [ns] / 비행 시간")
ax.set_ylabel("Counts / 카운트")
ax.set_title("Simulated LENA ITOF spectrum (Fig. 11 analog) / LENA ITOF 스펙트럼 모의")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def composition_image(
    base_img: np.ndarray, h_fraction_map: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Split a LENA image into H and O channels using a per-pixel H fraction.

    During a storm, cleft and dayside pixels are O-rich (heavy ion outflow);
    polar cap is H-rich (classic polar wind).

    Args:
        base_img: Total image in arbitrary flux units, shape (n_polar, n_az).
        h_fraction_map: Optional per-pixel H fraction; if None, built from a
            simple model that decreases H fraction toward the dayside cleft.

    Returns:
        Tuple (h_image, o_image).
    """
    n_polar, n_az = base_img.shape
    if h_fraction_map is None:
        az = np.linspace(0, 2 * np.pi, n_az, endpoint=False)
        # Day side (az ~ pi) -> O-dominated; night side -> H-dominated.
        f_h_az = 0.5 + 0.4 * np.cos(az)
        f_h_pol = np.linspace(0.3, 0.9, n_polar)  # poleward = more H
        h_fraction_map = np.outer(f_h_pol, f_h_az)
        h_fraction_map = np.clip(h_fraction_map, 0.05, 0.95)
    h_img = base_img * h_fraction_map
    o_img = base_img * (1.0 - h_fraction_map)
    return h_img, o_img


h_img, o_img = composition_image(lena_image)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [lena_image, h_img, o_img],
    ["Total ENA / 전체 ENA", "H channel / 수소 채널", "O channel / 산소 채널"],
):
    im = ax.imshow(img, origin="lower", aspect="auto",
                   norm=LogNorm(vmin=1e6, vmax=img.max()), cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Azimuth pixel")
    ax.set_ylabel("Polar pixel")
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
plt.tight_layout()
plt.show()

ratio = h_img.sum() / o_img.sum()
print(f"Integrated H/O flux ratio (storm scenario): {ratio:.2f}")
print("In real storms, O+ outflow can exceed H+, especially near the cleft.")

## Part 4: Low-Energy Noise Model and Dayglow Rejection / 저에너지 잡음 모델 및 Dayglow 차단

LENA's main noise sources are:
1. **Photodesorbed negative ions** from the conversion-surface adsorbate layer (driven by Ly-α photon flux).
2. **Electronic / radiation singles** on the MCPs (~1–10 Hz).
3. **Random TOF coincidences** scaling as `R1 * R2 * t_dead`.

Lab measurements at 10⁻⁶ Torr give an upper bound of 20 counts s⁻¹ pixel⁻¹ at 10 kR Lyman-α; on-orbit at 10⁻⁸ Torr the rate drops by ~100×. We implement a parametric noise model and compute the SNR of the storm image.

주된 잡음원: Lα에 의한 photodesorbed 음이온, MCP 라디에이션, random coincidence.

In [ ]:
def background_rate_per_pixel(
    lyman_alpha_kr: float = 10.0, vacuum_torr: float = 1e-8
) -> float:
    """Background count rate per pixel from photodesorption + electronic noise.

    Model: scaling from the lab upper bound of 20 c/s/pixel at 10 kR Ly-a and
    1e-6 Torr, falling proportionally with pressure (less adsorbate replenishment
    means less material to photodesorb).

    Args:
        lyman_alpha_kr: Geocoronal Ly-a brightness in kilo-Rayleighs.
        vacuum_torr: Operating pressure in torr.

    Returns:
        Total background rate in counts s^-1 pixel^-1.
    """
    photodesorption = 20.0 * (lyman_alpha_kr / 10.0) * (vacuum_torr / 1e-6)
    electronic = 0.5  # fixed electronic floor (Hz)
    return photodesorption + electronic


def random_coincidence_rate(
    r1_hz: float = 100.0, r2_hz: float = 100.0, t_dead_ns: float = 300.0
) -> float:
    """Random coincidence rate (Eq. 1 of Moore et al. 2000)."""
    return r1_hz * r2_hz * (t_dead_ns * 1e-9)


vacua = np.logspace(-9, -6, 100)
rates_dim = [background_rate_per_pixel(2.0, p) for p in vacua]
rates_storm = [background_rate_per_pixel(15.0, p) for p in vacua]

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(vacua, rates_dim, label="Quiet Ly-a (2 kR) / 정온")
ax.loglog(vacua, rates_storm, label="Storm Ly-a (15 kR) / 폭풍")
ax.axvspan(1e-9, 1e-8, alpha=0.15, color="green", label="On-orbit / 궤도")
ax.axvspan(1e-7, 1e-6, alpha=0.15, color="red", label="Lab / 실험실")
ax.set_xlabel("Vacuum [torr] / 진공도")
ax.set_ylabel("Background rate [c/s/pixel] / 배경 카운트율")
ax.set_title("LENA background vs. vacuum and Ly-a / LENA 배경 vs 진공/Lα")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Random coincidence rate (R1=R2=100 Hz, t=300 ns): "
      f"{random_coincidence_rate():.2e} Hz")
print("Negligible contribution to the noise floor in normal operation.")

### Add noise to the storm image and compute SNR / 폭풍 영상에 잡음 추가 및 SNR 계산

Per-pixel signal counts during 120-s spin = `flux_at_pixel * A_eff * solid_angle * T_int`. We Poisson-sample both signal and background.

120 s 적분 동안 신호 + Poisson 배경을 sampling.

In [ ]:
def synthesize_observation(
    flux_image: np.ndarray, energy_ev: float = 100.0,
    species: str = "O", t_int_s: float = 120.0,
    solid_angle_sr: float = 0.02, lyman_alpha_kr: float = 15.0,
    vacuum_torr: float = 1e-8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate Poisson realization of a LENA observation.

    Args:
        flux_image: Source flux in cm^-2 s^-1 sr^-1 mapped to LENA pixels.
        energy_ev: Energy of the bin being imaged.
        species: "H" or "O".
        t_int_s: Integration time in seconds (one spin = 120 s).
        solid_angle_sr: Solid angle per pixel (~0.02 sr).
        lyman_alpha_kr: Geocoronal Ly-a brightness.
        vacuum_torr: Operating vacuum.

    Returns:
        (signal_counts, background_counts, total_counts) all shape == flux_image.
    """
    a_eff = effective_area(energy_ev, species)
    expected_signal = flux_image * a_eff * solid_angle_sr * t_int_s
    bg_rate = background_rate_per_pixel(lyman_alpha_kr, vacuum_torr)
    expected_bg = bg_rate * t_int_s * np.ones_like(flux_image)
    sig = RNG.poisson(np.maximum(expected_signal, 0))
    bg = RNG.poisson(expected_bg)
    return sig, bg, sig + bg


# Use the O channel of the storm image, scaled to a physical flux level.
flux_o = o_img / o_img.max() * 5e3  # peak ~ 5e3 cm^-2 s^-1 sr^-1
sig_o, bg_o, total_o = synthesize_observation(
    flux_o, energy_ev=300.0, species="O", lyman_alpha_kr=15.0
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [sig_o, bg_o, total_o],
    ["Signal (O) / 신호", "Background / 배경", "Observed / 관측"],
):
    im = ax.imshow(
        img, origin="lower", aspect="auto",
        vmin=0, vmax=max(total_o.max(), 1), cmap="plasma",
    )
    ax.set_title(title)
    ax.set_xlabel("Azimuth pixel")
    ax.set_ylabel("Polar pixel")
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
plt.tight_layout()
plt.show()

snr = sig_o.sum() / np.sqrt(max(bg_o.sum(), 1))
print(f"Image-integrated SNR (O channel, 300 eV bin, 120 s): {snr:.1f}")
print("This SNR is sufficient to recover the cleft + auroral oval morphology.")

## Summary / 요약

We reproduced the four core aspects of LENA's design and operation:

1. **Surface conversion** — A power-law `eta(E)` model captures the behavior of polished W with adsorbates: ~10⁻⁵ at 30 eV up to ~10⁻³ at 1 keV, with retained-energy fractions 0.80 (H) and 0.60 (O).
2. **Imaging geometry** — A synthetic auroral oval + cleft + polar cap source projects onto the 12×45 LENA pixel grid; polar PSF FWHM of 2.5 bins reproduces the angular smearing.
3. **H/O composition** — ITOF separates species via $t = L\sqrt{m/(2qV_{\rm post})}$ at 20 kV, yielding $t_O / t_H = 4$ for clean discrimination; the storm scenario shows O-rich cleft, H-rich polar cap.
4. **Noise** — Background scales linearly with vacuum and Ly-α brightness; on-orbit 10⁻⁸ Torr operation suppresses the lab-measured 20 c/s/pixel by ≥100×, enabling high-SNR storm imaging.

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 동등물 |
|---|---|---|
| Surface conversion / 표면 변환 | Polished W + adsorbates, η ~ 10⁻³ | IBEX-Lo, IMAP-Lo (same lineage) |
| ENA imaging cadence / ENA 영상 주기 | 120 s spin | TWINS dual-spacecraft stereo |
| H/O ITOF / H/O 비행 시간 | M/δM ≈ 4 at 20 kV | Hot-plasma instruments on MMS, IMAP |
| Geocoronal noise / 지콜로나 잡음 | Multi-stage UV rejection | Same paradigm; tighter blackening |

네 가지 핵심 측면 재현: 표면 변환, 영상 기하, H/O 구분, 잡음 모델. LENA의 설계 패러다임은 IBEX-Lo, IMAP-Lo로 직접 계승되었다.